<a href="https://colab.research.google.com/github/snehapradeep25/-Internship-Project---NPOL-/blob/main/CNN_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Libraries

In [ ]:
!pip install librosa tensorflow matplotlib seaborn scikit-learn

import numpy as np
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


# PATHS


import zipfile
zipfile.ZipFile('/content/drive/MyDrive/Project/vtuad_dataset.zip').extractall('/content/vtuad/')

DATASET_ROOT = '/content/vtuad/inclusion_2000_exclusion_4000/'
TRAIN_PATH = os.path.join(DATASET_ROOT, 'train')
VAL_PATH = os.path.join(DATASET_ROOT, 'validation')
TEST_PATH = os.path.join(DATASET_ROOT, 'test')

MODEL_SAVE_PATH = '/content/drive/MyDrive/Project/cnn_model/'
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

LOADING DATASET

In [ ]:
def get_file_lists(split_path):
    """Get audio file paths and labels"""
    files = []
    labels = []

    for class_idx, class_name in enumerate(['background', 'cargo', 'passengership', 'tanker', 'tug']):
        class_dir = os.path.join(split_path, 'audio', class_name)

        if os.path.exists(class_dir):
            for audio_file in os.listdir(class_dir):
                if audio_file.endswith('.wav'):
                    files.append(os.path.join(class_dir, audio_file))
                    labels.append(class_idx)

    return np.array(files), np.array(labels)

print("Loading file lists...")
train_files, train_labels = get_file_lists(TRAIN_PATH)
val_files, val_labels = get_file_lists(VAL_PATH)
test_files, test_labels = get_file_lists(TEST_PATH)

print(f" Train: {len(train_files)} files")
print(f" Val: {len(val_files)} files")
print(f" Test: {len(test_files)} files")

print("\nClass distribution (Train):")
for i, name in enumerate(['background', 'cargo', 'passengership', 'tanker', 'tug']):
    count = (train_labels == i).sum()
    print(f"  {name}: {count}")

FEATURE EXTRACTION

In [ ]:
def extract_mel_spectrogram(file_path, sr=32000, n_mels=128,
                            n_fft=2048, hop_length=512, duration=1.0):
    """Extract mel spectrogram with ref=1.0"""
    try:
        audio, _ = librosa.load(file_path, sr=sr, duration=duration)

        target_length = int(sr * duration)
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)), mode='constant')
        else:
            audio = audio[:target_length]

        mel_spec = librosa.feature.melspectrogram(
            y=audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
        )

        mel_spec_db = librosa.power_to_db(mel_spec, ref=1.0)

        return mel_spec_db

    except Exception as e:
        return None


CONFIG = {
    'sample_rate': 32000,
    'duration': 1.0,
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'batch_size': 64,
    'epochs': 25,
    'learning_rate': 0.0005,
    'patience': 20
}

NUM_CLASSES = 5

GLOBAL NORMALIZATION

In [ ]:
sample_size = 1000
sample_indices = np.random.choice(len(train_files), size=sample_size, replace=False)

all_mels = []
for idx in tqdm(sample_indices, desc="Extracting features"):
    mel = extract_mel_spectrogram(
        train_files[idx],
        sr=CONFIG['sample_rate'],
        n_mels=CONFIG['n_mels'],
        n_fft=CONFIG['n_fft'],
        hop_length=CONFIG['hop_length'],
        duration=CONFIG['duration']
    )
    if mel is not None:
        all_mels.append(mel)

all_mels = np.array(all_mels)
GLOBAL_MEAN = all_mels.mean(axis=(0, 2), keepdims=True)  # (1, 128, 1)
GLOBAL_STD = all_mels.std(axis=(0, 2), keepdims=True)    # (1, 128, 1)

print(f"\nComputed from {len(all_mels)} samples")
print(f"Mean shape: {GLOBAL_MEAN.shape}")
print(f"Mean range: [{GLOBAL_MEAN.min():.2f}, {GLOBAL_MEAN.max():.2f}]")
print(f"Std shape: {GLOBAL_STD.shape}")
print(f"Std range: [{GLOBAL_STD.min():.2f}, {GLOBAL_STD.max():.2f}]")

np.save('/content/drive/MyDrive/Project/cnn_model/global_mean.npy', GLOBAL_MEAN)
np.save('/content/drive/MyDrive/Project/cnn_model/global_std.npy', GLOBAL_STD)
print("\nSaved to Drive: global_mean.npy, global_std.npy")

DATA GENERATOR

In [ ]:
def data_generator_cnn(file_paths, labels, batch_size, mean, std, shuffle=True):
    """Generate batches for CNN"""
    num_samples = len(file_paths)

    while True:
        indices = np.arange(num_samples)
        if shuffle:
            np.random.shuffle(indices)

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]

            batch_files = file_paths[batch_indices]
            batch_labels = labels[batch_indices]

            batch_features = []
            batch_targets = []

            for file_path, label in zip(batch_files, batch_labels):
                mel_spec = extract_mel_spectrogram(
                    file_path, sr=CONFIG['sample_rate'], n_mels=CONFIG['n_mels'],
                    n_fft=CONFIG['n_fft'], hop_length=CONFIG['hop_length'],
                    duration=CONFIG['duration']
                )

                if mel_spec is not None:
                    mel_norm = (mel_spec - mean) / std
                    batch_features.append(mel_norm)
                    batch_targets.append(label)

            if len(batch_features) > 0:
                X = np.array(batch_features)
                X = np.expand_dims(X, axis=-1)
                y = keras.utils.to_categorical(batch_targets, num_classes=NUM_CLASSES)
                yield X, y

CNN MODEL ARCHITECTURE

In [ ]:
def build_cnn_model(input_shape=(128, 63, 1), num_classes=5):
    """CNN architecture with 4 convolutional blocks"""

    inputs = layers.Input(shape=input_shape)

    # Conv Block 1
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    # Conv Block 2
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    # Conv Block 3
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    # Conv Block 4
    x = layers.Conv2D(256, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    # Dense layers
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs, name='CNN_Model')

# Build
cnn_model = build_cnn_model(input_shape=(128, 63, 1), num_classes=NUM_CLASSES)

# Compile
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("="*70)
print("CNN MODEL ARCHITECTURE")
print("="*70)
cnn_model.summary()

print(f"\nTotal parameters: {cnn_model.count_params():,}")

TRAINING SETUP

In [ ]:
GLOBAL_MEAN = np.load('/content/drive/MyDrive/Project/cnn_model/global_mean.npy').reshape(128, 1)
GLOBAL_STD = np.load('/content/drive/MyDrive/Project/cnn_model/global_std.npy').reshape(128, 1)

print(f"Loaded normalization: mean={GLOBAL_MEAN.shape}, std={GLOBAL_STD.shape}")

steps_per_epoch_train = len(train_files) // CONFIG['batch_size']
steps_per_epoch_val = len(val_files) // CONFIG['batch_size']

train_gen = data_generator_cnn(train_files, train_labels, CONFIG['batch_size'],
                                GLOBAL_MEAN, GLOBAL_STD, shuffle=True)
val_gen = data_generator_cnn(val_files, val_labels, CONFIG['batch_size'],
                              GLOBAL_MEAN, GLOBAL_STD, shuffle=False)

print(f"Steps/epoch: train={steps_per_epoch_train}, val={steps_per_epoch_val}")

CNN_CHECKPOINT_DIR = '/content/drive/MyDrive/Project/cnn_model/checkpoints/'
os.makedirs(CNN_CHECKPOINT_DIR, exist_ok=True)

checkpoint_best = callbacks.ModelCheckpoint(
    filepath=os.path.join(CNN_CHECKPOINT_DIR, 'cnn_best.keras'),
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['patience'],
    restore_best_weights=True,
    verbose=1
)

csv_logger = callbacks.CSVLogger(
    os.path.join(CNN_CHECKPOINT_DIR, 'cnn_training_log.csv')
)

print("="*70)
print("STARTING CNN TRAINING")
print("="*70)

history_cnn = cnn_model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch_train,
    epochs=CONFIG['epochs'],
    validation_data=val_gen,
    validation_steps=steps_per_epoch_val,
    callbacks=[checkpoint_best, early_stop, csv_logger],
    verbose=1
)

print("\nCNN TRAINING COMPLETE")
cnn_model.save(os.path.join(CNN_CHECKPOINT_DIR, 'cnn_final.keras'))
print(f"Saved: cnn_final.keras")

EVALUATION

In [ ]:
cnn_model = keras.models.load_model(
    os.path.join(CNN_CHECKPOINT_DIR, 'cnn_best.keras')
)

print(f"Loaded: {CNN_CHECKPOINT_DIR}cnn_best.keras")
print(f"Input shape: {cnn_model.input_shape}")
print(f"Output shape: {cnn_model.output_shape}")

# Test generator
test_gen = data_generator_cnn(test_files, test_labels, CONFIG['batch_size'],
                               GLOBAL_MEAN, GLOBAL_STD, shuffle=False)

# Collect predictions
print("\nGenerating predictions on test set...")

y_true = []
y_pred = []
y_pred_proba = []

for X_batch, y_batch in test_gen:
    if len(y_true) >= len(test_files):
        break

    preds = cnn_model.predict(X_batch, verbose=0)
    y_pred_proba.extend(preds)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(y_batch, axis=1))

y_true = np.array(y_true[:len(test_files)])
y_pred = np.array(y_pred[:len(test_files)])
y_pred_proba = np.array(y_pred_proba[:len(test_files)])

print(f"Collected {len(y_pred)} predictions")

# Overall metrics
print("\n" + "="*70)
print("OVERALL TEST PERFORMANCE")
print("="*70)

test_accuracy = (y_pred == y_true).mean()
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")

# Per-class metrics
print("\n" + "="*70)
print("PER-CLASS METRICS")
print("="*70)

class_names = ['background', 'cargo', 'passengership', 'tanker', 'tug']
print("\n" + classification_report(y_true, y_pred,
                                    target_names=class_names, digits=4))

# Per-class breakdown
print("\nDetailed Per-Class Breakdown:")
for i, class_name in enumerate(class_names):
    class_mask = (y_true == i)
    class_total = class_mask.sum()
    class_correct = ((y_true == i) & (y_pred == i)).sum()
    class_acc = class_correct / class_total if class_total > 0 else 0
    print(f"\n{class_name.upper()}:")
    print(f"   Total samples: {class_total}")
    print(f"   Correct predictions: {class_correct}")
    print(f"   Accuracy: {class_acc*100:.2f}%")

# Confusion matrix
print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title(f'CNN Test Set Confusion Matrix\n(Accuracy: {test_accuracy*100:.2f}%)',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Project/cnn_model/cnn_confusion_matrix.png', dpi=150)
plt.show()
print("Saved: cnn_confusion_matrix.png")

# Error analysis
print("\n" + "="*70)
print("ERROR ANALYSIS")
print("="*70)

error_indices = np.where(y_pred != y_true)[0]
print(f"\nTotal misclassified: {len(error_indices)} "
      f"({len(error_indices)/len(y_true)*100:.2f}%)")

print("\nMost Common Misclassifications:")
from collections import Counter
error_pairs = list(zip(y_true[error_indices], y_pred[error_indices]))
error_counts = Counter(error_pairs)

for (true_cls, pred_cls), count in error_counts.most_common(10):
    print(f"   {class_names[true_cls]:15s} -> {class_names[pred_cls]:15s}: {count} times")


print("CNN EVALUATION COMPLETE")